# Facial Recognition

## Import

In [ ]:
import os
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Any, TypeAlias

import cv2 as cv
import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import display


In [ ]:
from deepface import DeepFace
from deepface.modules.exceptions import FaceNotDetected


## Settings

In [ ]:
PRERECORDING_PATH = Path('data/recording.avi')

if 'SSH_CLIENT' in os.environ:
	print('Remote session detected. Using video file.')
	VIDEO_SOURCE = PRERECORDING_PATH
else:
	print('Local session detected. Using live webcam.')
	VIDEO_SOURCE = 0

# Make sure to start the postgresql daemon
os.environ['DEEPFACE_POSTGRES_URI'] = 'postgresql://postgres:@localhost/deepface'

RECORDING_FPS=30.0
OUTPUT_PATH = Path('data/output.avi')

RECOGNITION_MODEL = 'Facenet'
DISTANCE_METRIC = 'euclidean_l2'

DB_PATH = Path('data/faces_db')
FACE_YOUNGER_PATH = Path('data/faces_db/Dylan/younger.jpg')
FACE_OLDER_PATH = Path('data/faces_db/Dylan/peak.jpg')
FACE_RECENT_PATH = Path('data/faces_db/Dylan/webcam_smirk.png')


## Generic CV Function

In [ ]:
# Over-engineered to all hell
class VideoProcessor:

	@dataclass
	class DisplayConfig:
		pass

	@dataclass
	class DisplayOption_Headless(DisplayConfig):
		pass

	@dataclass
	class DisplayOption_Jupyter(DisplayConfig):
		image_widget: widgets.Image

	@dataclass
	class DisplayOption_OpenCV(DisplayConfig):
		pass

	#DisplayConfig: TypeAlias = DisplayOption_Headless | DisplayOption_Jupyter | DisplayOption_OpenCV

	FrameCallbackType: TypeAlias = Callable[[cv.typing.MatLike], cv.typing.MatLike | None]

	_END_LOOP = False
	_GO_AGAIN = True

	@staticmethod
	def _display_video(
		display_option: DisplayConfig,
		frame: cv.typing.MatLike
	) -> bool:
		"""Will attempt to display an output based on the display option."""

		match display_option:
			case VideoProcessor.DisplayOption_Jupyter(image_widget=image_widget):
				ret: bool
				buffer: np.ndarray[Any, np.dtype[np.uint8]]
				ret, buffer = cv.imencode(ext='.jpg', img=frame)

				if not ret:
					print("Can't encode frame as image. Exiting ...")
					return VideoProcessor._END_LOOP

				image_widget.value = buffer.tobytes()

			case VideoProcessor.DisplayOption_OpenCV:
				print('Frame should be displayed')
				cv.imshow(winname='frame', mat=frame)

		return VideoProcessor._GO_AGAIN

	@staticmethod
	def _frame_loop(
		vid_cap: cv.VideoCapture,
		callback: FrameCallbackType,
		display_option: DisplayConfig,
		frametime: float
	) -> bool:
		"""This function is called continuously until either the video ends, or is interrupted."""

		start_time: float = time.time()

		ret: bool
		frame: cv.typing.MatLike
		ret, frame = vid_cap.read()

		if not ret:
			print("Can't receive frame (stream end?). Exiting ...")
			return VideoProcessor._END_LOOP

		frame_result: cv.typing.MatLike | None = callback(frame)
		if frame_result is None: frame_result = frame

		if not VideoProcessor._display_video(
			display_option=display_option,
			frame=frame_result
		): return VideoProcessor._END_LOOP

		match display_option:
			case VideoProcessor.DisplayOption_OpenCV:
				if cv.waitKey(delay=int(1/frametime)) == ord('q'):
					return VideoProcessor._END_LOOP

			case _: # VideoProcessor.DisplayOption_Jupyter:
				# Enforce the framerate pacing
				elapsed_time: float = time.time() - start_time
				time_to_wait: float = frametime - elapsed_time
				if time_to_wait > 0: time.sleep(time_to_wait)

		return VideoProcessor._GO_AGAIN

	@staticmethod
	def process_video(
		capture_location: int | str | Path,
		/,
		callback: FrameCallbackType = lambda x: x,
		*,
		display_config: DisplayConfig | None = None,
		frametime: float = 1.0 / 30.0
	) -> None:
		"""Read from a video input and apply the callback to it.

		Args:
			capture_location (int | str | Path): What the video source is.
				- For a webcam input, use an int.
				- For a video file, use a str or Path.
			callback (FrameCallbackType: Callable[[cv.typing.MatLike], cv.typing.MatLike | None]): Function that will be called with each frame.
			frametime (float): In seconds, how long between each frame. Use 1 / fps if you want to pass in a framerate.
			display_type (DisplayType): Whether to output the frames, and if so, how should it be shown. Options:
				- Jupyter Notebook.
				- Qt window via OpenCV.

		Returns:
			None:
		"""

		# Null sentinel
		if display_config is None:
			display_config = VideoProcessor.DisplayOption_Jupyter(
				# JPEG is faster than PNG
				image_widget=widgets.Image(format='jpeg')
			)

		vid_cap = cv.VideoCapture(capture_location)

		if not vid_cap.isOpened():
			print('Cannot open video source.')
			return

		if isinstance(display_config, VideoProcessor.DisplayOption_Jupyter):
			display(display_config.image_widget)

		try:
			while VideoProcessor._frame_loop(
				vid_cap=vid_cap,
				callback=callback,
				display_option=display_config,
				frametime=frametime
			): pass

		except KeyboardInterrupt:
			print('Video stream interrupted.')

		finally:
			vid_cap.release()


## Testing Deepface

### Prebuild Model

In [ ]:
DeepFace.build_model(RECOGNITION_MODEL)


### Face Verification

In [ ]:
result: dict = DeepFace.verify(
	img1_path=str(FACE_YOUNGER_PATH),
	img2_path=str(FACE_OLDER_PATH),
	model_name=RECOGNITION_MODEL,
	distance_metric=DISTANCE_METRIC,
	normalization=RECOGNITION_MODEL
)
display(result)


In [ ]:
def convert_path_to_string(path: Path, /) -> str:
	# There is a strong temptation to turn this into a single statement, but it would compromise readability.
	all_suffixes = ''.join(path.suffixes)
	base_name = path.name.removesuffix(all_suffixes)
	return '_'.join(path.parts[:-1] + (base_name,))

	# Alternative that does looping stuff
	#while path.suffix:
	#	path = path.with_suffix('')
	#return '_'.join(path.parts)


In [ ]:
def register(
	path: Path | list[Path],
	/,
	model: str = RECOGNITION_MODEL
) -> int:
	if not isinstance(path, list):
		path = [path]

	sum = 0

	for p in path:
		result: dict[str, int] = DeepFace.register(
			img=str(p),
			img_name=convert_path_to_string(p),
			model_name=model,
			normalization=model
		)
		sum += result['inserted']
	return sum


In [ ]:
# Add faces to the database
register([FACE_YOUNGER_PATH, FACE_OLDER_PATH, FACE_RECENT_PATH])


In [ ]:
DeepFace.build_index(RECOGNITION_MODEL)


In [ ]:
dfs: list[pd.DataFrame] = DeepFace.search(
	img=str(FACE_RECENT_PATH),
	model_name=RECOGNITION_MODEL,
	normalization=RECOGNITION_MODEL
)

for df in dfs:
	display(df)


### Face Verification In Video

In [ ]:
def embed_face(face, /) -> list[float]:
	represent_dict: list[dict[str, Any]] | list[list[dict[str, Any]]] = DeepFace.represent(
		img_path=face,
		model_name=RECOGNITION_MODEL,
		normalization=RECOGNITION_MODEL
	)

	# Match the type
	match represent_dict[0]:
		case list() as inner_list:
			return inner_list[0]['embedding']
		case dict() as inner_dict:
			return inner_dict['embedding']
		case _:
			raise TypeError("Unexpected data structure.")


In [ ]:
face_embedding: list[float] = embed_face(str(FACE_RECENT_PATH))


In [ ]:
def verify_against_image(frame: cv.typing.MatLike):
	try:
		# Will throw an exception if no face is found
		frame_embedding: list[float] = embed_face(frame)

		result: dict = DeepFace.verify(
			img1_path=frame_embedding,
			img2_path=face_embedding,
			model_name=RECOGNITION_MODEL,
			distance_metric=DISTANCE_METRIC,
			normalization=RECOGNITION_MODEL
		)
		display(result)

	except FaceNotDetected:
		print('No face found.')


In [ ]:
VideoProcessor.process_video(
	VIDEO_SOURCE,
	callback=verify_against_image,
	frametime=1
)


## Output Display

### Display Video Source

In [ ]:
# Will display webcam when run locally, or video file when run remotely
VideoProcessor.process_video(VIDEO_SOURCE, lambda frame: cv.cvtColor(frame, cv.COLOR_RGB2GRAY))


### Display Prerecorded Video

In [ ]:
# Will always display the video file
VideoProcessor.process_video(
	PRERECORDING_PATH,
	callback=lambda frame: cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
)


### Record Webcam

In [ ]:
# Define the codec and create VideoWriter object
fourcc = cv.VideoWriter_fourcc(*'XVID') # type: ignore
out = cv.VideoWriter(
	filename=OUTPUT_PATH,
	fourcc=fourcc,
	fps=RECORDING_FPS,
	frameSize=(640,  480)
)

VideoProcessor.process_video(
	0,
	callback=lambda frame: out.write(frame),
	frametime=1.0 / RECORDING_FPS
)

out.release()


### DeepFace Auto Face Detection

In [ ]:
DeepFace.stream(
	db_path=str(DB_PATH / 'Dylan'),
	model_name=RECOGNITION_MODEL,
	distance_metric=DISTANCE_METRIC,
	enable_face_analysis=False,
	source=VIDEO_SOURCE,
	time_threshold=0,
	frame_threshold=0,
	output_path=str(OUTPUT_PATH)
)
